In [ ]:
import pywt
import scipy.signal as sp
import numpy as np
import plotly.graph_objects as go

## Testing convolution and trimming

In [ ]:
def get_trim_len(filter_len, mode="valid"):
    """
    Get the length to trim off both ends of a full convolution to convert it to a valid
    or same transform.
    """
    if mode == "valid":
        lim = (filter_len - 1)
        return lim, lim
    elif mode == "same":
        lim1 = (filter_len - 1) // 2
        offset = 1 if filter_len % 2 == 0 else 0
        lim2 = lim1 + offset
        return lim1, lim2
    else:
        raise ValueError("""mode must be in ["valid", "same"]""")

def data_keep(data, filter_len, mode):
    lim1, lim2 = get_trim_len(filter_len, mode)
    return data[lim1:-lim2]

def downsample(data):
    lim = 2 * (len(data) // 2)
    return data[1:lim:2]

errors = 0
mode = "same"

for i in range(50, 100):
    for j in range(10, 30):
        data = np.random.randn(i)
        filter = np.random.randn(j)

        data_conv = sp.convolve(data, filter, "full")

        data_reduced = sp.convolve(data, filter, mode)

        data_trim = data_keep(data_conv, len(filter), mode)
        errors += abs(sum(data_reduced - data_trim))

print(errors)

In [ ]:
def get_trim_len(filter_len, level, mode="valid"):
    """
    Get length to trim off both ends of a downsammpled full convolution to convert it to a
    downsampled valid or same convolution.
    """
    scale_factor = 2 * (1 - 2**-level)  # the additional length of signal to
    if mode == "valid":
        trim_val = filter_len - 1
    elif mode == "same":
        trim_val = (filter_len - 1) // 2
    else:
        raise ValueError("""mode must be in ["valid", "same"]""")
    return int((trim_val * scale_factor) // 2)  # Downsampled data is trimmed less

def data_keep(data, filter_len, mode, levels):
    lim1 = get_trim_len(filter_len, levels, mode)
    return data[lim1:-lim1]

def downsample(data):
    lim = 2 * (len(data) // 2)
    return data[1:lim:2]

levels = 5
mode = "valid"

data = np.random.randn(5000)
filter = np.random.randn(30)

data_conv = data.copy()
for _ in range(levels):
    data_conv = sp.convolve(data_conv, filter, "full")
    data_conv = downsample(data_conv)

data_reduced = data.copy()
for _ in range(levels):
    data_reduced = sp.convolve(data_reduced, filter, mode)
    data_reduced = downsample(data_reduced)

data_trim = data_keep(data_conv, len(filter), mode, levels)
print(f"data with full convolution: {len(data_conv)}")
print(f"trimmed data: {len(data_trim)}")
print(f"data with {mode} convolution: {len(data_reduced)}")

error = sum(np.array(data_reduced) - np.array(data_trim))
print(error)

x = [i for i in range(len(data_reduced))]

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=x, y=data_reduced)
)
fig.add_trace(
    go.Scatter(x=x, y=data_trim, line=dict(dash="solid"))
)
fig.show()

print(error)

## Inverse transform

In [ ]:
def upsample(signal):
    upsampled_signal = np.zeros(len(signal) * 2)

    # Fill the upsampled signal with the original values
    upsampled_signal[1::2] = signal
    return upsampled_signal

In [ ]:
t = np.arange(0, 20, .1)
signal = np.random.randn(len(t))

In [ ]:
wavelet = "db10"

filter_bank = pywt.Wavelet(wavelet).filter_bank
down_coeffs = filter_bank[0]
down_coeffs_approx = filter_bank[1]
up_coeffs = filter_bank[2]
up_coeffs_approx = filter_bank[3]

convolved_signal = sp.convolve(signal, down_coeffs, mode="full")
cs2 = sp.convolve(signal, down_coeffs_approx, mode="full")

downsampled_signal = convolved_signal[1::2]
ds2 = cs2[1::2]

upsampled_signal = upsample(downsampled_signal)
us2 = upsample(ds2)

rec_signal = sp.convolve(upsampled_signal, up_coeffs, mode="valid")
rs2 = sp.convolve(us2, up_coeffs_approx, mode="valid")

full_rec = rec_signal + rs2

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=t, y=signal, name="signal")
)
fig.add_trace(
    go.Scatter(x=t, y=full_rec, name="rec signal", line=dict(dash="dash"))
)
fig.update_layout(
    title="signal vs rec single"
)

In [ ]:
wavelet = "db30"

filter_bank = pywt.Wavelet(wavelet).filter_bank
down_coeffs = filter_bank[0]
down_coeffs_approx = filter_bank[1]
up_coeffs = filter_bank[2]
up_coeffs_approx = filter_bank[3]

convolved_signal = sp.filtfilt(down_coeffs, 1, signal)
cs2 = sp.filtfilt(down_coeffs_approx, 1, signal)

downsampled_signal = convolved_signal[::2]
ds2 = cs2[::2]

upsampled_signal = upsample(downsampled_signal)
us2 = upsample(ds2)

rec_signal = sp.filtfilt(up_coeffs, 1, upsampled_signal)
rs2 = sp.filtfilt(up_coeffs_approx, 1, us2)

full_rec = rec_signal + rs2
full_rec = full_rec  # [len(down_coeffs) - 1:]

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=t, y=signal, name="signal")
)
fig.add_trace(
    go.Scatter(x=t, y=1/2*(convolved_signal+cs2), name="rec signal", line=dict(dash="dash"))
)
fig.update_layout(
    title="signal vs rec single"
)